# Generates Trial Info Databases for Each Mouse

In [2]:
import os
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
from scipy.io import loadmat


# ----------------------------
# Paths (new)
# ----------------------------
ANALYSIS_ROOT = Path(r"Y:/Voltage/VisualConsciousness/Analysis/Visual")
BEHAV_ROOT = Path(r"Y:/Voltage/VisualConsciousness/BehaviorDataBackup - Copy/VoltageMice")


def behavior_path(mouse: str) -> Path:
    # .../VoltageMice/<mouse>/PerceptualData_<mouse>_all.mat
    return BEHAV_ROOT / mouse / f"PerceptualData_{mouse}_all.mat"


def volt_results_path(mouse: str, date_YYMMDD: int | str) -> Path:
    # .../VoltageMice/<mouse>/Volt_<date>.mat
    return BEHAV_ROOT / mouse / f"mxvolt_{int(date_YYMMDD)}_processed.mat"


def movie_path(mouse: str, date_YYMMDD: int | str, meas_folder: str) -> Path:
    # Y:/.../Analysis/Visual/<mouse>/20<date>/<measXX>/cG_unmixed_dFF.h5
    return ANALYSIS_ROOT / mouse / f"20{int(date_YYMMDD)}" / meas_folder / "cG_unmixedTR_dFF.h5"


# ----------------------------
# Helpers from your old code
# ----------------------------
def flatten_cell(cell):
    if isinstance(cell, (list, np.ndarray)):
        return cell[0] if len(cell) > 0 else None
    return cell

def extract_specs(filepath):
    """Extracts FPS, time origin, and movie length from an HDF5 file."""
    with h5py.File(filepath, 'r') as mov_file:
        specs = mov_file["specs"]
        fps = specs["fps"][()].squeeze()
        timeorigin = specs["timeorigin"][()].squeeze()
        movie_length = mov_file["mov"].shape[0]
    return fps, timeorigin, movie_length

def compute_delays_and_bfm_times(df):
    """ Computes delays and BFM times for each unique (Date, AnimalCode, Recording) combination, adds columns to df """

    # Force scalar types so pandas doesn't create weird row objects on some days
    df = df.copy()
    df["Date"] = df["Date"].apply(lambda x: int(np.asarray(x).squeeze()))
    df["Recording"] = df["Recording"].apply(lambda x: int(np.asarray(x).squeeze()))
    df["AnimalCode"] = df["AnimalCode"].apply(lambda x: str(np.asarray(x).squeeze()))

    delay_cache = {}
    specs_cache = {}

    unique_pairs = df[["Date", "AnimalCode", "Recording", "File"]].drop_duplicates()

    for _, row in unique_pairs.iterrows():
        date = row["Date"]
        mouse = row["AnimalCode"]
        recording = int(row["Recording"])  # ensure int
        file = row["File"]

        # Compute delay
        path2 = volt_results_path(mouse, date)
        data2 = loadmat(path2)
        df2 = pd.DataFrame(data2["MasterN"]).map(lambda x: x[0] if isinstance(x, np.ndarray) and x.size > 0 else x)
        df2.columns = df2.iloc[0]
        df2 = df2[1:]
        delayEstimates = df2["FrameAlignmentInfo"][recording][0][0]
        delay = float(np.mean(delayEstimates))
        delay_cache[(date, mouse, recording)] = delay

        # Extract movie specs
        filepath = movie_path(mouse, date, file)
        fps, timeorigin, movie_length = extract_specs(filepath)
        specs_cache[(date, mouse, recording)] = (fps, timeorigin, movie_length)

    # --- MINIMAL: avoid df.apply for Delay (robust + faster) ---
    keys = list(zip(df["Date"], df["AnimalCode"], df["Recording"]))
    df["Delay"] = [delay_cache.get(k, np.nan) for k in keys]

    def _to_float(x):
        # robust scalar extraction for numpy scalars/arrays/lists
        x = np.asarray(x).squeeze()
        return float(x)

    def _bfm_time(row):
        key = (row["Date"], row["AnimalCode"], int(row["Recording"]))
        fps, timeorigin, movie_length = specs_cache.get(key, (np.nan, np.nan, np.nan))

        try:
            t = _to_float(row["Time"])
            d = _to_float(row["Delay"])
            fps = _to_float(fps)
            timeorigin = _to_float(timeorigin)
        except Exception:
            return np.nan

        return t - d - (timeorigin / fps)

    df["BFMTime"] = [_bfm_time(row) for _, row in df.iterrows()]

    def _valid_trial(row):
        key = (row["Date"], row["AnimalCode"], int(row["Recording"]))
        fps, timeorigin, movie_length = specs_cache.get(key, (np.nan, np.nan, np.nan))
        try:
            bfm = _to_float(row["BFMTime"])
            fps = _to_float(fps)
            movie_length = int(np.asarray(movie_length).squeeze())
        except Exception:
            return False

        movie_dur_s = movie_length / fps
        return bool((0 <= bfm) and (bfm <= movie_dur_s))

    df["ValidTrial?"] = [_valid_trial(row) for _, row in df.iterrows()]

    return df


def perceptual_category(df: pd.DataFrame, data_behavior: dict) -> pd.DataFrame:
    percept_cats = pd.DataFrame(data_behavior["ConCrit"])
    min_values = percept_cats.iloc[1:, 0].to_list()
    max_values = percept_cats.iloc[1:, 1].to_list()
    bins = [min_values[0]] + max_values

    categories = pd.cut(
        df["Contrast"],
        bins=bins,
        labels=[1, 2, 3],
        right=True,
        include_lowest=True,
    )

    if "PerceptualCat" in df.columns:
        df = df.drop(columns=["PerceptualCat"])
    df.insert(5, "PerceptualCat", categories)
    return df


def categorize_trial(row):
    if (row["ReactTime"] < 0.25) or (row["Enticed?"] == 1) or ((row["Rewarded?"] == 1) and (row["Consume?"] == 0)):
        return "Error (0)"

    if row["Rewarded?"] == 1 and row["Attrition?"] == 0:
        if row["PerceptualCat"] == 1:
            return "False Alarm (1)"
        elif row["PerceptualCat"] == 2:
            return "MC Hit (2)"
        elif row["PerceptualCat"] == 3:
            return "HC Hit (3)"

    if row["Rewarded?"] == 0 and row["Attrition?"] == 0:
        if row["PerceptualCat"] == 1:
            return "Correct Rejection (4)"
        elif row["PerceptualCat"] == 2:
            return "MC Miss (5)"
        elif row["PerceptualCat"] == 3:
            return "Incorrect Reject (6)"

    if row["Rewarded?"] == 0 and row["Attrition?"] == 1:
        if row["PerceptualCat"] == 1:
            return "LC No Report (7)"
        elif row["PerceptualCat"] == 2:
            return "MC No Report (8)"
        elif row["PerceptualCat"] == 3:
            return "HC No Report (9)"

    return "Uncategorized"


def trial_category(df: pd.DataFrame, print_results: bool = False) -> pd.DataFrame:
    df = df.copy()
    df.loc[(df["Attrition?"] == 1) & (df["Rewarded?"] == 1), "Attrition?"] = 0
    df["TrialType"] = df.apply(categorize_trial, axis=1)

    if print_results:
        print("\nTrial Type Summary:")
        summary = df["TrialType"].value_counts()
        for category, count in summary.items():
            print(f"{category}: {count} trials")

    return df


def trial_ID(df: pd.DataFrame, mouse: str) -> pd.DataFrame:
    df = df.copy()
    df["TrialID"] = (
        "Visual/"
        + str(mouse)
        + "/20"
        + df["Date"].astype(str)
        + "/"
        + df["File"].astype(str)
        + "/trial"
        + df["Trial"].astype(str).str.zfill(3)
    )
    return df


# ----------------------------
# New: per-day generator
# ----------------------------
def _load_trialinfo_df(mouse: str) -> tuple[pd.DataFrame, dict]:
    data_behavior = loadmat(behavior_path(mouse), struct_as_record=False, squeeze_me=True)

    df = pd.DataFrame(data_behavior["TrialInfo"]).map(flatten_cell).map(flatten_cell)
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)

    # normalize column naming like your old code
    if "Recording#" in df.columns and "Recording" not in df.columns:
        df = df.rename(columns={"Recording#": "Recording"})
    return df, data_behavior


def _recording_to_meas_map(mouse: str, date_YYMMDD: int | str) -> list[int] | None:
    """
    Returns file_numbers list indexed by (Recording-1). These are the numbers from RESULTS[:, col 5].
    """
    try:
        mat_data = loadmat(f'Y:/Voltage/VisualConsciousness/BehaviorDataBackup - Copy/VoltageMice/{mouse}/mxvolt_{int(date_YYMMDD)}', struct_as_record=False, squeeze_me=True)
        results = mat_data["RESULTS"]
        file_numbers = results[1:, 4]  # column 5 (index 4)
        return list(file_numbers)
    except Exception as e:
        print(f"Error loading Volt_{int(date_YYMMDD)}.mat for {mouse}: {e}")
        return None


def gen_trial_info_day(
    mouse: str,
    date_YYMMDD: int | str,
    require_movie: bool = True,
    print_results: bool = True,
) -> pd.DataFrame:
    """
    Temporary simpler day-only trial info.

    - Filters behavioral TrialInfo to the requested date
    - Adds PerceptualCat + TrialType
    - Maps Recording -> measXX using Volt_<date>.mat (RESULTS)
    - Optionally keeps only trials whose movie file exists
    """
    date_YYMMDD = int(date_YYMMDD)

    df, data_behavior = _load_trialinfo_df(mouse)

    # Filter to this day
    df = df[df["Date"].astype(int) == date_YYMMDD].copy()
    df.reset_index(drop=True, inplace=True)

    if df.empty:
        return df

    # Add categories
    df = perceptual_category(df, data_behavior)
    df = trial_category(df, print_results=print_results)

    # Map Recording -> measXX
    file_numbers = _recording_to_meas_map(mouse, date_YYMMDD)
    if not file_numbers:
        # still return categorized df, just without File/paths
        return df

    def rec_to_meas(rec: int) -> str | None:
        if pd.isna(rec):
            return None
        rec = int(rec)
        if rec < 1 or rec > len(file_numbers):
            return None
        # match your old code: subtract 1 then format as meas0X (but do it robustly)
        idx = int(file_numbers[rec - 1]) - 1
        if idx < 0:
            return None
        return f"meas{idx:02d}"

    df["File"] = df["Recording"].apply(rec_to_meas)

    # Add movie path + filter to existing movies if desired
    df["MoviePath"] = df["File"].apply(lambda f: str(movie_path(mouse, date_YYMMDD, f)) if isinstance(f, str) else None)

    if require_movie:
        exists_mask = df["MoviePath"].apply(lambda p: isinstance(p, str) and Path(p).is_file())
        df = df[exists_mask].copy()
        df.reset_index(drop=True, inplace=True)

    # TrialID (consistent with your new movie path)
    df = trial_ID(df, mouse)
    df = compute_delays_and_bfm_times(df)

    # Suggested compact column order (keeps extras too)
    preferred = [
        "TrialID", "TrialType",
        "AnimalCode", "Date", "Recording", "File",
        "Trial", "Time", "BFMTime", "Duration", "Contrast", "PerceptualCat",
        "ReactTime", "Rewarded?", "Enticed?", "Seen?", "Consume?", "Attrition?",
        "EngagementScore", "EngagementScore_S20", "VDT Behavior Quality",
        "MoviePath"
    ]
    cols = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]
    df = df[cols]

    return df

def gen_trial_info_mouse(
    mouse: str,
    require_movie: bool = True,
    print_results: bool = True,
) -> pd.DataFrame:
    """
    Generate one large trial-info dataframe for ALL days for a given mouse.

    - Loops over all unique days in PerceptualData_<mouse>_all.mat
    - Calls gen_trial_info_day for each day
    - Concatenates results
    - If require_movie=True, trials from recordings missing the movie file are skipped
      (because gen_trial_info_day already filters by MoviePath existence).

    Returns
    -------
    pd.DataFrame
        Concatenated trial info across all available days/recordings for this mouse.
    """
    # Load once to get the list of days (minimal overhead, avoids guessing folder structure)
    df_all, _ = _load_trialinfo_df(mouse)
    days = sorted(df_all["Date"].astype(int).unique())

    dfs = []
    for d in days:
        df_day = gen_trial_info_day(
            mouse=mouse,
            date_YYMMDD=d,
            require_movie=require_movie,
            print_results=False,   # avoid spamming per-day summaries
        )
        if df_day is not None and not df_day.empty:
            dfs.append(df_day)

    if len(dfs) == 0:
        # Return empty df with expected columns if nothing found
        return pd.DataFrame(columns=df_all.columns)

    df = pd.concat(dfs, ignore_index=True)

    if print_results and "TrialType" in df.columns:
        print("\nTrial Type Summary (ALL DAYS COMBINED):")
        summary = df["TrialType"].value_counts()
        for category, count in summary.items():
            print(f"{category}: {count} trials")

    return df

In [3]:
df_mouse = gen_trial_info_mouse("cfm006mjr", require_movie=True, print_results=True)


Trial Type Summary (ALL DAYS COMBINED):
HC Hit (3): 114 trials
MC Miss (5): 61 trials
Correct Rejection (4): 56 trials
MC Hit (2): 37 trials
Incorrect Reject (6): 31 trials
HC No Report (9): 25 trials
Error (0): 23 trials
False Alarm (1): 10 trials
MC No Report (8): 8 trials
LC No Report (7): 5 trials


In [4]:
df_mouse.to_csv('trial_info/TrialInfo_cfm006mjr.csv')